# Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu prediksi omzet harian tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi total_penjualan sebuah pesanan berdasarkan atribut-atributnya.
Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
import spark_session

app = spark_session.SparkApp("Tugas11-MLlib")
spark = app.getSession()

Memulai SparkSession untuk Tugas11-MLlib


26/11/16 09:07:33 WARN Utils: Your hostname, asfadani resolves to a loopback address: 127.0.1.1; using 192.168.18.43 instead (on interface wlx088af1bd9610)
26/11/16 09:07:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 09:07:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [3]:
df = spark.createDataFrame(data_tugas11)
df.printSchema()
print("Total baris:", df.count())

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- total_penjualan: double (nullable = true)



[Stage 0:>                                                        (0 + 12) / 12]

Total baris: 2500


### A. Feature Engineering (bobot 20%)

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.

**1. Melakukan StringIndexer**

In [40]:
from pyspark.ml.feature import StringIndexer

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx",)
indexer_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

indexer_menu_model = indexer_menu.fit(df)
indexer_cabang_model = indexer_cabang.fit(df)
indexer_pesan_model = indexer_pesan.fit(df)
df_indexed = indexer_menu_model.transform(df)
df_indexed = indexer_cabang_model.transform(df_indexed)
df_indexed = indexer_pesan_model.transform(df_indexed)

df_indexed.select("nama_menu", "cabang", "waktu_pesan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx").show(5)

+--------------+----------+-----------+-------------+----------+---------------+
|     nama_menu|    cabang|waktu_pesan|nama_menu_idx|cabang_idx|waktu_pesan_idx|
+--------------+----------+-----------+-------------+----------+---------------+
|Kentang Goreng|  Semarang|      Malam|          5.0|       2.0|            1.0|
|Kentang Goreng|Yogyakarta|       Sore|          5.0|       1.0|            3.0|
|Kentang Goreng|  Semarang|      Siang|          5.0|       2.0|            2.0|
|Kentang Goreng|Yogyakarta|       Pagi|          5.0|       1.0|            0.0|
|    Roti Bakar|  Magelang|      Malam|          4.0|       0.0|            1.0|
+--------------+----------+-----------+-------------+----------+---------------+
only showing top 5 rows



**2. Melakukan VectorAssembler**

In [7]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "qty", "harga_satuan"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[5.0,2.0,1.0,1.0,30000.0]|27750.163531558544|
|[5.0,1.0,3.0,3.0,20000.0]|64216.16074371864 |
|[5.0,2.0,2.0,5.0,22000.0]|108722.79259154423|
|[5.0,1.0,0.0,2.0,20000.0]|54483.205871792496|
|[4.0,0.0,1.0,2.0,30000.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



### B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

**1. Split data**

In [8]:
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

Jumlah data training : 1825
Jumlah data testing   : 675


**2. Melatih model LinearRegression**

In [10]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

model = lr.fit(train)
print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Model berhasil dilatih!
Koefisien tiap fitur: [-85.74425311498636,206.62345418252357,281.87593287908965,21899.69969532968,3.007052812619656]
Intercept: -65835.73823558415


**3. Membuat prediksi**

In [14]:
predictions = model.transform(test)
predictions.select("nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "qty", "harga_satuan", "total_penjualan", "prediction").show(5)

+-------------+----------+---------------+---+------------+------------------+-----------------+
|nama_menu_idx|cabang_idx|waktu_pesan_idx|qty|harga_satuan|   total_penjualan|       prediction|
+-------------+----------+---------------+---+------------+------------------+-----------------+
|          0.0|       0.0|            2.0|  5|       30000| 158075.1016480395|134438.0964854121|
|          0.0|       0.0|            3.0|  1|       20000|12588.856856324866|17050.64551077591|
|          0.0|       2.0|            0.0|  2|       15000| 30243.34367205522| 23482.7002527351|
|          0.0|       2.0|            0.0|  2|       25000| 42515.77413473763|53553.22837893166|
|          0.0|       2.0|            0.0|  3|       22000| 70935.47683487965|66431.76963640236|
+-------------+----------+---------------+---+------------+------------------+-----------------+
only showing top 5 rows



### C. Evaluasi Model (bobot 20%)

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,670
R²   : 0.9131  (91.3% variasi data berhasil dijelaskan model)


>Melihat dari evaluasi model tersebut yang skor r2 nya bisa mencapai 91%, menjadikan model sudah sangat cukup baik untuk dipakai memprediksi omzet.

### D. Eksperimen Perbandingan Fitur (bobot 25%)

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [19]:
from pyspark.ml.feature import VectorSlicer

slicer = VectorSlicer(
    inputCol="features", 
    outputCol="features2", 
    indices=[3, 4] # Mengambil qty (indeks 3) dan harga_satuan (indeks 4)
)

df_final2 = slicer.transform(df_final)
df_final2.select("features2", "total_penjualan").show(5, truncate=False)

+-------------+------------------+
|features2    |total_penjualan   |
+-------------+------------------+
|[1.0,30000.0]|27750.163531558544|
|[3.0,20000.0]|64216.16074371864 |
|[5.0,22000.0]|108722.79259154423|
|[2.0,20000.0]|54483.205871792496|
|[2.0,30000.0]|58157.94412015849 |
+-------------+------------------+
only showing top 5 rows



In [26]:
train2, test2 = df_final2.randomSplit([0.75, 0.25], seed=101)

lr2 = LinearRegression(featuresCol="features2", labelCol="total_penjualan")
model2 = lr2.fit(train2)

predictions2 = model2.transform(test2)
predictions2.select("qty", "harga_satuan", "total_penjualan", "prediction").show(5)

evaluator_rmse2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse2 = evaluator_rmse2.evaluate(predictions2)

evaluator_r22 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r22 = evaluator_r22.evaluate(predictions2)

print(f"RMSE : Rp {rmse2:,.0f}")
print(f"R²   : {r22:.4f}  ({r22*100:.1f}% variasi data berhasil dijelaskan model)")

+---+------------+------------------+------------------+
|qty|harga_satuan|   total_penjualan|        prediction|
+---+------------+------------------+------------------+
|  5|       30000| 158075.1016480395|  134249.088865085|
|  1|       20000|12588.856856324866|16639.040166945888|
|  2|       15000| 30243.34367205522|   23516.436355227|
|  2|       25000| 42515.77413473763|53549.924336898555|
|  3|       22000| 70935.47683487965| 66434.01812151397|
+---+------------+------------------+------------------+
only showing top 5 rows

RMSE : Rp 10,680
R²   : 0.9129  (91.3% variasi data berhasil dijelaskan model)


>Dilihat dari hasil evaluasinya, menunjukkan bahwa menggunakan hanya fitur qty dan harga_satuan, malah memberikan hasil akurasi yang lebih tinggi. Tapi perbedaanya sangat kecil sehingga tidak berarti.

### E. Prediksi Skenario Baru (bobot 15%)

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.

In [29]:
data_dummy = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Es Teh"],
    "cabang": ["Yogyakarta", "Magelang", "Semarang"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [2, 1, 3],
    "harga_satuan": [20000, 30000, 15000]
})

spark_df = spark.createDataFrame(data_dummy)

In [41]:
# Indexer
df_idx = indexer_menu_model.transform(spark_df)
df_idx = indexer_cabang_model.transform(df_idx)
df_idx = indexer_pesan_model.transform(df_idx)

df_idx.select("nama_menu", "cabang", "waktu_pesan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx").show(3)

# vectorassembler
assembler = VectorAssembler(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "qty", "harga_satuan"],
    outputCol="features"
)
df_features = assembler.transform(df_idx)

+----------+----------+-----------+-------------+----------+---------------+
| nama_menu|    cabang|waktu_pesan|nama_menu_idx|cabang_idx|waktu_pesan_idx|
+----------+----------+-----------+-------------+----------+---------------+
| Kopi Susu|Yogyakarta|       Pagi|          1.0|       1.0|            0.0|
|Roti Bakar|  Magelang|      Siang|          4.0|       0.0|            2.0|
|    Es Teh|  Semarang|      Malam|          3.0|       2.0|            1.0|
+----------+----------+-----------+-------------+----------+---------------+



In [42]:
# 3. Prediksi menggunakan model yang sudah dilatih sebelumnya
prediksi_skenario = model.transform(df_features)
# 4. hasil prediksi
prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show()

+----------+----------+-----------+---+------------+-----------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|       prediction|
+----------+----------+-----------+---+------------+-----------------+
| Kopi Susu|Yogyakarta|       Pagi|  2|       20000|38225.59660853587|
|Roti Bakar|  Magelang|      Siang|  1|       30000|46496.32069163343|
|    Es Teh|  Semarang|      Malam|  3|       15000|45407.04312159891|
+----------+----------+-----------+---+------------+-----------------+



In [44]:
app.stopSession()

Session Tugas11-MLlib sudah dihentikan
